# **Selección de hiperparámetros del tracker OC-SORT**

Este documento tiene las siguientes partes:

- **¿Qué es OC-SORT?**: Explica el algoritmo de seguimiento
- **Características de la selección**: Explica el modelo de detección utilizado, la implementación de OC-SORT utilizada, y las referencias utilizadas para la búsqueda de hiperparámetros
- **Grid de hiperparámetros**: Explica qué hiperparámetros se prueban y por qué.
- **Pruebas realizadas**: Presenta el código utilizado para seguir a los jugadores
- **Análisis de hiperparámetros**: Tiene 3 partes:
    - **Obtención de métricas**: Presenta el código utilizado para utilizar el seguimiento obtenido con la sección anterior y obtener las métricas.
    - **Obtención de gráficas**: Presenta el código que utiliza las métricas para hacer las gráficas que se analizan
    - **Análisis de gráficas**: Análisis de los hiperparámetros.

## **¿Qué es OC-SORT?**

[OC-SORT](https://arxiv.org/abs/2203.14360) trata de mejorar el seguimiento de objetos que siguen movimientos no lineales, cambios bruscos de movimiento, y oclusiones.

Se emplea un filtro de Kalman por cada identidad para predecir dónde se encuentra en el siguiente frame, prediciendo la posición de cada identidad y actualizando su posición vista en el frame actual cuando esta se asocia con una detección.

Se añade un paso para los objetos que no se han asociado en un cierto número de frames y se han vuelto a asociar. Para ello, se crea una trayectoria virtual de los frames perdidos teniendo en cuenta la trayectoria pasada del objeto y su posición actual, y se actualiza el estado de la identidad usando esta trayectoria, con el fin de no acumular error en el filtro de Kalman.

OC-SORT tiene en cuenta tanto el IoU de la detección candidata y la predicción de Kalman, como el ángulo de diferencia de las velocidades que sigue la identidad en frames anteriores y la detección candidata. De esta forma, se usan ambas medidas para obtener el coste de la asociación.

Si se calcula la velocidad utilizando pocos frames, la velocidad es más ruidosa, pero si se calcula utilizando muchos frames, se asume que el movimiento del objeto es lineal durante ese periodo de tiempo, lo que es erróneo en un número elevado de frames al tratarse de objetos que siguen movimientos no lineales. Por lo tanto, OC-SORT hace un balance entre estas dos opciones.


## **Características de la selección**

- **Modelo de detección de jugadores**: Se utiliza el modelo YOLO entrenado para detectar jugadores de baloncesto en  mi Trabajo Fin de Grado.
- **Implementación de OC-SORT**: Se utiliza la librería BoxMOT.
- **Referencias**: En vez de empezar de cero con la búsqueda de hiperparámetros, se toma como referencia los valores de los parámetros utilizados en [este trabajo](https://arxiv.org/abs/2209.13154), en el que se utiliza el tracker OC-SORT para el seguimiento de jugadores de baloncesto, así como otros dos deportes, pero los hiperparámetros se escogen independientemente para cada deporte. Una de las diferencias del trabajo con el de este repositorio es que tiene en cuenta el número de dorsal y el equipo para corregir los cambios de identidad. En este notebook, sin embargo, se usa OC-SORT sin modificaciones, por lo que las métricas obtenidas no son directamente comparables con las del trabajo de referencia, ya que se evalúa sobre un conjunto distinto, se consideran los tres deportes en lugar de uno solo y no se aplica la limitación de 10 identidades máximas utilizada allí.

## **Grid de hiperparámetros**

- **max_age**: Número de frames que se mantiene una identidad sin detección en los siguientes frames. El valor en el trabajo previamente nombrado es de 30, por lo que se prueban el valor 30, junto con 60 y 120 para observar si un mayor número de frames mejora el resultado. No se prueban valores menores a 30 porque, dado que es un partido de baloncesto, es común que un jugador se vea ocluido durante más de un segundo, y dado que los vídeos no suelen tener menos de 30 FPS, un valor menor a 30 no se considera adecuado para este contexto.
- **iou_threshold**: IoU mínimo para considerar que una detección corresponde con una identidad predicha. El valor en el trabajo previamente nombrado es de 0.3, por lo que se prueban el valor 0.3, junto con un valor menor (0.2) y un valor mayor (0.5).
- **det_thresh**: Valor de confianza en la detección mínima para tener en cuenta esa detección. Como en el trabajo en el que se entrenó el modelo de detección se obtuvieron buenos resultados con un valor de 0.6, se prueban valores alrededor de 0.6, en concreto se prueban los valores 0.5, 0.55, 0.6, 0.65, y 0.7.

## **Pruebas realizadas**

Las pruebas realizadas se hicieron usando scripts de Python. A continuación, se explica el código del script.

En primer lugar estas son las librerías usadas en el script:

In [ ]:
# Para manejar rutas de archivos
import os 
import glob
from pathlib import Path

# Para leer imágenes
import cv2 

# Para manejo de arrays
import numpy as np

# Para detección de los jugadores
from ultralytics import YOLO

# Implementación de OCSORT
from boxmot import OcSort 


A continuación, la función **procesa_carpeta** procesa la secuencia de imágenes de la carpeta donde se encuentran los frames de la secuencia y  guarda su resultado en carpeta_salida/nombre_de_secuencia.txt.

In [ ]:
# Definir las constantes
NOMBRE_MODELO = "Nombre con el que se ha guardado el modelo entrenado, por ejemplo 'deteccion_jugadores.pt'"
DEVICE = "cuda:0"  # Si no se tiene GPU, cambiar a cpu
REID_MODEL = "osnet_x0_25_msmt17.pt" # Modelo de re-identificación de jugadores a utilizar

def procesa_carpeta(carpeta_secuencia, carpeta_salida, det_thresh, iou_threshold, max_age):
    """Procesa la carpeta img1/ de carpeta_secuencia, donde se encuentran los frames de la secuencia, 
    y guarda su resultado en carpeta_salida/nombre_de_secuencia.txt en el formato indicado 
    (https://codalab.lisn.upsaclay.fr/competitions/12424#learn_the_details-submission)"""
    
    # Accede a la carpeta de imágenes
    carpeta_imagenes = Path(carpeta_secuencia) / "img1"
    if not carpeta_imagenes.exists():
        print(f"Error: No existe {carpeta_imagenes}")
        return
    
    # Se queda con el path al fichero de salida
    nombre_secuencia = Path(carpeta_secuencia).name
    fichero_salida = Path(carpeta_salida) / f"{nombre_secuencia}.txt"
    
    # Modelo
    model = YOLO(NOMBRE_MODELO)
    
    # Configuración ocsort
    tracker = OcSort(
        det_thresh=det_thresh,
        max_age=max_age,
        iou_threshold=iou_threshold,
        device=DEVICE,
        model_weights=REID_MODEL
    )
    
    # Ordena los nombres de path a frames para acceder a ellos de forma ordenada
    tracks = []
    paths_frames = sorted(carpeta_imagenes.glob("*.jpg"), key=lambda x: int(x.stem))
    
    print(f"Procesando la secuencia {nombre_secuencia} ({len(paths_frames)} frames)")
    
    for frame_idx, img_path in enumerate(paths_frames, 1):
        # Accede al frame
        frame = cv2.imread(str(img_path))
        if frame is None: continue # Por si acaso hay errores
            
        # Detecta jugadores (clase 0)
        resultado_deteccion = model(frame, classes=[0], conf=det_thresh, verbose=False)
        
        # Si se han detectado jugadores
        if resultado_deteccion[0].boxes is not None and len(resultado_deteccion[0].boxes) > 0:
            # Detecciones
            detecciones = np.array(resultado_deteccion[0].boxes.data.tolist())
            
            # Actualiza tracker (obtiene tracks en frame actual)
            tracks_actual = tracker.update(detecciones, frame)
            
            # Si hay tracks en frame actual, los guarda en el objeto tracks
            if tracks_actual is not None:
                for track in tracks_actual:
                    # Obtiene los datos para almacenar en el fichero en el formato correcto
                    bb_left, bb_top, bb_right, bb_bottom, track_id, conf, _, _ = track
                    bb_width = int(bb_right - bb_left)
                    bb_height = int(bb_bottom - bb_top)
                    
                    # <frame>, <id>, <bb_left>, <bb_top>, <bb_width>, <bb_height>, <conf>, <x>, <y>, <z>
                    # The world coordinates x,y,z are ignored for the 2D challenge and can be filled with -1
                    line = [frame_idx, int(track_id), int(bb_left), int(bb_top), bb_width, bb_height, float(conf), -1, -1, -1]
                    tracks.append(line)
    
    # Crea la carpeta de salida si no está creada
    os.makedirs(carpeta_salida, exist_ok=True)
    # Guarda la información en el fichero de salida
    with open(fichero_salida, 'w') as f:
        for track in tracks:
            f.write(f"{track[0]} {track[1]} {track[2]} {track[3]} {track[4]} {track[5]} {track[6]} {track[7]} {track[8]} {track[9]}\n")
    
    print(f"Procesada secuencia {nombre_secuencia} ({fichero_salida})")

La función guarda la información del seguimiento de jugadores en una carpeta indicada, que después se usa para obtener las métricas sobre cómo de bien funciona el conjunto de hiperparámetros probado.

En el *main* del código, se establecen las carpetas donde guardar los resultados y los parámetros a probar. Una vez almacenado en las carpetas correspondientes, se utiliza TrackEval para obtener las métricas de seguimiento y poder comparar las combinaciones de los hiperparámetros.

## **Análisis de hiperparámetros**

Se analiza el rendimiento de los hiperparámetros en el conjunto de entrenamiento de SportsMOT.

### **Obtención de las métricas**

Se obtienen las métricas desde la carpeta correspondiente (uso de TrackEval). Para ello, se utiliza el siguiente código

In [ ]:
import pandas as pd
import os

# Constantes
METRICAS = ["IDSW", "IDs", "GT_IDs", "IDTP", "IDFN", "IDFP", "IDF1", "AssA"] # Métricas que nos interesan
CARPETA_RESULTADOS = "Nombre de la carpeta donde se encuentran los resultados"
INICIO_NOMBRE_PRUEBAS_OCSORT = "Nombre por el que empiezan todas las pruebas de hiperparámetros que se probaron para OCSORT" # Las pruebas también tienen los valores de los hiperparémtros en el nombre de la prueba para que se puedan extraer y analizar posteriormente

def obtener_metricas(carpeta_pruebas):
    if not os.path.exists(carpeta_pruebas):
        print(f"Error: No se encuentra la carpeta {carpeta_pruebas}")
        return
    
    metricas_pruebas = []
    for carpeta_prueba in os.listdir(carpeta_pruebas):

        # Obtenemos el fichero
        path_summary = os.path.join(os.path.join(CARPETA_RESULTADOS, carpeta_prueba), "pedestrian_summary.txt")
        fichero_summary = pd.read_csv(path_summary, sep=r'\s+')

        # Obtenemos las métricas relevantes
        metricas_prueba = fichero_summary[METRICAS].copy()
        metricas_prueba["Modelo"] = carpeta_prueba
        metricas_pruebas.append(metricas_prueba)

    df_metricas = pd.concat(metricas_pruebas, ignore_index=True)
    return df_metricas

def obtener_metricas_ocsort(carpeta_pruebas):
    metricas = obtener_metricas(carpeta_pruebas)
    metricas_ocsort = metricas[metricas["Modelo"].str.startswith(INICIO_NOMBRE_PRUEBAS_OCSORT)]

    # Extraer IoU, conf y age del nombre del modelo
    def extraer_parametros(modelo):
        partes = modelo.split("_")
        conf = float(partes[5][4:])  # Extraer valor después de "conf"
        iou = float(partes[6][3:])   # Extraer valor después de "iou"
        age = int(partes[7][3:])     # Extraer valor después de "age"
        return conf, iou, age
    metricas_ocsort[["Conf", "IoU", "Age"]] = metricas_ocsort["Modelo"].apply(lambda x: pd.Series(extraer_parametros(x)))
    return metricas_ocsort

metricas_df = obtener_metricas_ocsort(CARPETA_RESULTADOS)

# Poner el índice de nuevo a 0, 1, 2, ...
metricas_df.reset_index(drop=True, inplace=True)

### **Obtención de gráficas**

Las gráficas se obtienen con el siguiente código. Es relevante indicar que se realiza lo mismo con "Conf", IoU" y "Age" para obtener las gráficas correspondientes a los tres hiperparámetros.

In [ ]:
df = metricas_df.copy()

# IDF1 medio por confianza
conf = df.groupby("Conf")["IDF1"].mean()

# Plotearlo con puntos
import matplotlib.pyplot as plt
plt.figure(figsize=(10, 6))
plt.plot(conf.index, conf.values, marker='o')
plt.title("IDF1 medio según la confianza del detector")
plt.xlabel("det_thresh")
plt.ylabel("IDF1 medio")
plt.grid()
plt.show()

# Mostrar boxplots de IDSW, IDs, IDTP, IDFN, IDFP, IDF1 y AssA para cada grupo de confianza del detector con 3 boxplots por fila. 
# Cada boxplot tiene una distribución distinta para el eje y
metricas = ["IDSW", "IDs", "IDTP", "IDFN", "IDFP", "IDF1", "AssA"]
for metrica in metricas:
    plt.figure(figsize=(10, 6))
    df.boxplot(column=metrica, by="Conf")
    plt.title(f"{metrica} según confianza del detector")
    plt.suptitle("")
    plt.xlabel("det_thresh")
    plt.ylabel(metrica)
    plt.grid()
    # Guardar la figura
    plt.savefig(f"Path a la carpeta deseada / det_thresh_{metrica}_boxplot.png", bbox_inches='tight')
    plt.show()



Por último, se imprime el modelo con mejor IDF1

In [ ]:
mejor_modelo = df.loc[df["IDF1"].idxmax()]
print("Mejor modelo:")
print(mejor_modelo)

### **Análisis de gráficas**

*A continuación, se analizan las gráficas con el mismo texto que se expuso en la memoria del proyecto. Las imágenes están repartidas un poco diferentes dado que no existen los labels automáticos, se reorganizan las imágenes para que se lean en el orden en el que son nombradas*

Dado que el grid de hiperparámetros da un número elevado de combinaciones, se muestran gráficos de la métrica IDF1 media para cada valor de cada hiperparámetro para poder visualizar el impacto de los valores de forma más clara.

#### **Confianza del detector**

En la siguiente figura se muestra el IDF1 medio por cada valor de confianza del detector.

![IDF1 medio por valor de confianza del detector](../../img/ocsort/det_thresh.png)

*IDF1 medio por valor de confianza del detector.*

Se observa que el valor de IDF1 aumenta aproximadamente 2 de media al pasar de 0.5 a 0.55. Sin embargo, los valores entre 0.55 y 0.65 tienen un impacto mucho menor, siendo el mejor valor medio el de confianza 0.65. Al aumentar la confianza a 0.7, el IDF1 medio disminuye. 

<table>
  <tr>
    <td align="center">
      <img src="../../img/ocsort/det_thresh_IDSW_boxplot.png" width="100%">
    </td>
    <td align="center">
      <img src="../../img/ocsort/det_thresh_IDs_boxplot.png" width="100%">
    </td>
    <td align="center">
      <img src="../../img/ocsort/det_thresh_IDTP_boxplot.png" width="100%">
    </td>
    <td align="center">
      <img src="../../img/ocsort/det_thresh_IDFN_boxplot.png" width="100%">
    </td>
  </tr>
  <tr>
    <td align="center">
      <img src="../../img/ocsort/det_thresh_IDFP_boxplot.png" width="100%">
    </td>
    <td align="center">
      <img src="../../img/ocsort/det_thresh_IDF1_boxplot.png" width="100%">
    </td>
    <td align="center">
      <img src="../../img/ocsort/det_thresh_AssA_boxplot.png" width="100%">
    </td>
    <td></td>
  </tr>
</table>

<p align="center"><em>Boxplots de las métricas para cada valor de confianza del detector</em></p>

Al analizar las métricas más detalladamente en la tabla de figuras anterior, una mayor confianza implica un menor número de IDSW, y el número de IDs disminuye de 0.5 a 0.65. Sin embargo, el número de IDs sube ligeramente de 0.65 a 0.7. Es positivo que el número de IDs disminuya dado que el número real de IDs es 150, y un número más cercano a este implica menor fragmentación de trayectorias. 

El número de IDTP es mayor en los valores entre 0.55 y 0.65, lo que indica un mayor número de frames en el que la trayectoria real y la predicha coinciden. El número de IDFN también es menor en el conjunto de valores entre 0.55 y 0.65. Sin embargo, el número de IDFP es menor cuanto mayor es la confianza.

Esto podría deberse a que si el número de detecciones consideradas es demasiado bajo (mayor confianza), el algoritmo de seguimiento no es tan capaz de detectar a los jugadores en todos los frames, lo que podría fragmentar algunas trayectorias en algunos casos y aumentar el número de IDs e IDSW. Sin embargo, un mayor número de detecciones consideradas (menor confianza) podría generar un mayor número de detecciones falsas. Por ejemplo, podría detectar al mismo jugador dos veces en el mismo frame, lo que podría fragmentar la trayectoria del jugador y generar un mayor número de IDFP. Por lo tanto, se debe encontrar un equilibrio.

#### **Umbral de IoU**

En la siguiente figura se muestra el IDF1 medio por cada valor de umbral de IoU.

![IDF1 medio por umbral de IoU](../../img/ocsort/iou_threshold.png)

*IDF1 medio por umbral de IoU.*

Se observa que el valor de 0.2 es ligeramente mejor que el valor de 0.3, con una diferencia aproximada de 1 punto en IDF1 medio. Sin embargo, si se aumenta a 0.5, el IDF1 medio disminuye considerablemente.

<table>
  <tr>
    <td align="center">
      <img src="../../img/ocsort/iou_IDSW_boxplot.png" width="100%">
    </td>
    <td align="center">
      <img src="../../img/ocsort/iou_IDs_boxplot.png" width="100%">
    </td>
    <td align="center">
      <img src="../../img/ocsort/iou_IDTP_boxplot.png" width="100%">
    </td>
    <td align="center">
      <img src="../../img/ocsort/iou_IDFN_boxplot.png" width="100%">
    </td>
  </tr>
  <tr>
    <td align="center">
      <img src="../../img/ocsort/iou_IDFP_boxplot.png" width="100%">
    </td>
    <td align="center">
      <img src="../../img/ocsort/iou_IDF1_boxplot.png" width="100%">
    </td>
    <td align="center">
      <img src="../../img/ocsort/iou_AssA_boxplot.png" width="100%">
    </td>
    <td></td>
  </tr>
</table>

<p align="center"><em>Boxplots de las métricas para cada valor de umbral de IoU</em></p>

Si se observan las métricas más detalladamente en la tabla de figuras anterior, el número de IDs e IDSW aumenta considerablemente, el número de IDTP disminuye y el de IDFP y IDFN aumenta, lo que podría deberse a que, al ser un umbral de IoU demasiado alto, el algoritmo descarta asociaciones correctas, lo que fragmenta las identidades y genera un mayor número de intercambios de identidad.

#### **Valor de max_age**

En la siguiente figura se muestra el IDF1 medio por cada valor de max_age. Para este valor, se debe encontrar un equilibrio entre mantener las identidades durante un número suficiente de frames para evitar fragmentar las trayectorias y no mantenerlas durante un número excesivo de frames por evitar aumentar el tiempo de computación y que se asocie una identidad incorrecta dado que no se conoce dónde se encuentra el jugador por no haberse detectado durante un número alto de frames.

![IDF1 medio por valor de max_age](../../img/ocsort/max_age.png)

*IDF1 medio por valor de max_age.*

Se observa en la figura que el valor con mejor IDF1 medio es de 60.

#### **Conjunto de hiperparámetros con mejor IDF1**

Por último, se mira el conjunto de hiperparámetros que mejor valor de IDF1 tiene, y este se trata de una confianza de 0.65, un umbral de IoU de 0.2, y un valor de max_age de 60. Por lo tanto, los valores por separado siguen la misma tendencia que el mejor conjunto de hiperparámetros.

A continuación, se obtiene el resultado de este conjunto de hiperparámetros en el conjunto de validación. El resultado se muestra en la siguiente tabla, que obtiene un IDF1 de 60.35. 

<table>
  <thead>
    <tr>
      <th>IDSW</th>
      <th>IDs</th>
      <th>GT_IDs</th>
      <th>IDTP</th>
      <th>IDFN</th>
      <th>IDFP</th>
      <th>IDF1</th>
      <th>AssA</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td>476</td>
      <td>421</td>
      <td>150</td>
      <td>69179</td>
      <td>49646</td>
      <td>41237</td>
      <td>60.35</td>
      <td>40.63</td>
    </tr>
  </tbody>
</table>

<p><em>Valores de métricas para OC-SORT con el mejor conjunto de hiperparámetros</em></p>